# Lab 04｜什麼是典型學生 Summary & Comparisons

<a href="https://colab.research.google.com/github/johnnychao/statistics-in-context-bilingual/blob/main/labs/colab/lab-04-summary-comparisons.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

> Statistics in Context · Unit 1 · 原創合成資料 · 不評量 Python 語法


## Goal

情境：學校考慮提供早餐，但先要描述「有吃早餐」與「沒吃早餐」兩群的學習準備度。

- 比較 mean/median 與 standard deviation/IQR。
- 使用 side-by-side boxplots 進行情境式群組比較。
- 寫出 **compare distributions（比較分布）**的 AP English 回答，不做因果宣稱。


## Setup

依序執行儲存格即可，不需要撰寫或背誦 Python。若想重新開始，請在 Colab 選擇 **Runtime → Restart session and run all**。

本 Lab 使用原創合成資料；所有代碼與數值均不對應真實學生。


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


In [ ]:
# 集中設定：一般情況只需修改這一格的參數。
DATA_RELATIVE_PATH = "data/public/morning_routine_survey.csv"
REPO_RAW_BASE_URL = "https://raw.githubusercontent.com/johnnychao/statistics-in-context-bilingual/main"

LOCAL_REPO_ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path("/content/statistics-in-context-bilingual"),
]


def load_repo_csv(relative_path):
    # 先找本機 repo，再讀 GitHub raw；失敗時提供繁中修復訊息。
    relative_path = Path(relative_path)
    for candidate_root in LOCAL_REPO_ROOT_CANDIDATES:
        candidate = candidate_root / relative_path
        if candidate.is_file():
            return pd.read_csv(candidate), str(candidate.resolve())

    remote_url = f"{REPO_RAW_BASE_URL}/{relative_path.as_posix()}"
    try:
        return pd.read_csv(remote_url), remote_url
    except Exception as exc:
        raise RuntimeError(
            "無法載入資料。請確認網路連線，或從 GitHub repo 根目錄執行此 Notebook。"
            f" 嘗試的遠端網址：{remote_url}。"
            " 若 repo 尚未發布，請先將 data/public 的 CSV 上傳至 main branch。"
        ) from exc


data, data_source = load_repo_csv(DATA_RELATIVE_PATH)
print(f"已載入 {len(data)} 筆資料｜Loaded {len(data)} rows")
print(f"來源 Source: {data_source}")


## Steps

### 1. 設定群組與結果變數

完成主要分析後，可把 `GROUP_VARIABLE` 改為 `arrival_status`，觀察三組比較。


In [ ]:
# ✏️ 修改任務：完成早餐比較後，嘗試 GROUP_VARIABLE = "arrival_status"。
GROUP_VARIABLE = "ate_breakfast"
MEASURE_VARIABLE = "readiness_score"

analysis_data = data[[GROUP_VARIABLE, MEASURE_VARIABLE]].dropna()
grouped = analysis_data.groupby(GROUP_VARIABLE)[MEASURE_VARIABLE]
summary = grouped.agg(
    n="count",
    mean="mean",
    median="median",
    sd="std",
    q1=lambda values: values.quantile(0.25),
    q3=lambda values: values.quantile(0.75),
).reset_index()
summary["iqr"] = summary["q3"] - summary["q1"]
display(summary.round(2))


### 2. 製作並閱讀並排箱形圖

箱形圖顯示 median、quartiles、IQR 與依規則標示的點；它不直接顯示每一組的樣本平均數。


In [ ]:
FIGURE_ALT = (
    "Side-by-side boxplots compare synthetic readiness scores for students who ate "
    "breakfast and students who did not, showing center, spread, and flagged points."
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(
    data=analysis_data,
    x=GROUP_VARIABLE,
    y=MEASURE_VARIABLE,
    order=sorted(analysis_data[GROUP_VARIABLE].unique()),
    color="#90CAF9",
    ax=ax,
)
sns.stripplot(
    data=analysis_data,
    x=GROUP_VARIABLE,
    y=MEASURE_VARIABLE,
    order=sorted(analysis_data[GROUP_VARIABLE].unique()),
    color="#23395B",
    alpha=0.35,
    size=3,
    jitter=0.25,
    ax=ax,
)
ax.set_title("Learning readiness by breakfast status (synthetic survey)")
ax.set_xlabel("Ate breakfast before school（是否吃早餐）")
ax.set_ylabel("Learning readiness score（points）")
plt.tight_layout()
plt.show()
print(f"Alt text: {FIGURE_ALT}")


### 3. 準備比較證據

當群組只有 `yes` 與 `no` 時，下格會計算 yes minus no。若你把群組改成三類，改用上方表格選兩組比較。


In [ ]:
if {"yes", "no"}.issubset(set(summary[GROUP_VARIABLE])):
    indexed_summary = summary.set_index(GROUP_VARIABLE)
    mean_difference = indexed_summary.loc["yes", "mean"] - indexed_summary.loc["no", "mean"]
    median_difference = indexed_summary.loc["yes", "median"] - indexed_summary.loc["no", "median"]
    print(f"Mean difference (yes − no) = {mean_difference:.2f} points")
    print(f"Median difference (yes − no) = {median_difference:.2f} points")
else:
    print("目前不是 yes/no 兩組；請從 summary 表選擇兩組比較。")


<details>
<summary><strong>AP English Response frame</strong></summary>

> The ____ group has a [higher/lower] typical readiness score: its median is ____ points compared with ____ points for ____. The ____ group also has [more/less/about the same] variability, with an IQR of ____ versus ____. Because this is an observational survey, the comparison does not show that breakfast causes the difference.

</details>

數位 FRQ 練習：用 4–6 句同時比較 center、variability、overlap/unusual features，並加入 observational study 的限制。


## Checks

確認各組筆數加總、IQR 與標準差合理。


In [ ]:
assert int(summary["n"].sum()) == len(analysis_data)
assert (summary["iqr"] >= 0).all()
assert (summary["sd"] >= 0).all()
print("✅ Checks passed：群組摘要涵蓋全部有效觀察值。")


## Next Steps

群組有差異，不代表問卷能代表全校。Lab 05 會把這 240 筆合成資料當作已知 population，比較 SRS、stratified sample 與 convenience/voluntary sample。
